# pandas.DataFrame - 2

Завантаження з csv-файлу та обробка відсутніх значень.<br>
Групування даних.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
print(sys.platform, sys.version, f'numpy version {np.__version__}',  f'pandas version {pd.__version__}',sep='\n')

win32
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
numpy version 2.4.1
pandas version 3.0.0


### Завантаження з csv-файлу та обробка відсутніх значень

Документація на функцію <code>read_csv</code>  <a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas-read-csv" target=_blank rel="noopener noreferrer">тут</a>.

In [2]:
data = pd.read_csv(Path('prepare', 'населення_plain2.csv'), sep=',', encoding='utf-8', header=None)
data.head(3)

,0,1,2
0,Київ,2001.0,2611327.0
1,Київ,2014.0,2868702.0
2,Київ,2020.0,2967360.0


In [3]:
data = pd.read_csv(Path('prepare', 'населення_plain2.csv'), sep=',', encoding='utf-8', 
                   header=None, names=['city', 'year', 'people'],
                   dtype={'year':'Int64', 'people':'Int64'}
                  )
data.head(3)

,city,year,people
0,Київ,2001,2611327
1,Київ,2014,2868702
2,Київ,2020,2967360


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   city    261 non-null    str  
 1   year    260 non-null    Int64
 2   people  260 non-null    Int64
dtypes: Int64(2), str(1)
memory usage: 6.8 KB


Можна бачити, що всього рядків 261, проте по деяких стовпчиках даних тільки 260. Це означає, що в деяких клітинах таблиці дані були відсутні.

При обробці даних з такими клітинами треба щось робити:
<ul>
<li>або заповнити якимось значенням</li>
<li>або вилучити відповідні рядки</li>
</lu>

Якби там не було, але перед прийняттям рішення щодо відсутніх значень дані спочатку треба дослідити.

Документація на метод <code>describe</code>  <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html#pandas.DataFrame.describe" target=_blank rel="noopener noreferrer">тут</a>.

In [5]:
data.describe()

,year,people
count,260.0,260.0
mean,2011.657692,238073.557692
std,7.959486,368780.320898
min,2001.0,40593.0
25%,2001.0,68181.75
50%,2014.0,103728.0
75%,2020.0,262676.25
max,2020.0,2967360.0


In [6]:
data.describe(include='all')

,city,year,people
count,261,260.0,260.0
unique,87,<NA>,<NA>
top,Київ,<NA>,<NA>
freq,3,<NA>,<NA>
mean,NaN,2011.657692,238073.557692
std,NaN,7.959486,368780.320898
min,NaN,2001.0,40593.0
25%,NaN,2001.0,68181.75
50%,NaN,2014.0,103728.0
75%,NaN,2020.0,262676.25


NA, N/A = not applicable, не застосовно<br>
Така позначка може зустрічатись у клітинах реальних таблиць, якщо дані відсутні.

Метод <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html" target=_blank rel="noopener noreferrer"><code>isna</code></a> повертає логічну ознаку відсутності даних у клітинах таблиці.<br>
Метод <code>pandas.DataFrame.any</code> аналогічний методу <code>numpy.ndarray.any</code>.

Щоб вибрати рядки, в яких є незаповнені клітини, вказуємо <code>axis=1</code>.

In [7]:
data[data.isna().any(axis=1)]

,city,year,people
4,Харків,<NA>,1451132
5,Харків,2020,<NA>


In [8]:
data[data['year'].isna() | data['people'].isna()]

,city,year,people
4,Харків,<NA>,1451132
5,Харків,2020,<NA>


Вилучаємо рядки, в яких не зазначено кількість мешканців.

In [9]:
data.dropna(subset='people', inplace=True)
data[data.isna().any(axis=1)]

,city,year,people
4,Харків,<NA>,1451132


Далі незаповнені клітини є тільки у стовпчику з роками. Заповнюємо відсутні значення значенням 2014.

In [10]:
data.fillna(2014, inplace=True)
data.info()

<class 'pandas.DataFrame'>
Index: 260 entries, 0 to 260
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   city    260 non-null    str  
 1   year    260 non-null    Int64
 2   people  260 non-null    Int64
dtypes: Int64(2), str(1)
memory usage: 8.6 KB


Тепер незаповнених клітин у таблиці нема, переведемо чисельність населення в тисячі та округлимо.

In [11]:
data['people'] = np.round(data['people']/1000, 1)
data.head(3)

,city,year,people
0,Київ,2001,2611.3
1,Київ,2014,2868.7
2,Київ,2020,2967.4


### Групування та групові(агрегатні) операції

Документація на метод <code>groupby</code>  <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html#pandas.DataFrame.groupby" target=_blank rel="noopener noreferrer">тут</a>.

In [12]:
data.groupby('year').sum()

,city,people
year,,
2001,КиївХарківОдесаДніпроДонецькЗапоріжжяЛьвівКрив...,21602.5
2014,КиївХарківОдесаДніпроДонецькЗапоріжжяЛьвівКрив...,20977.9
2020,КиївОдесаДніпроДонецькЗапоріжжяЛьвівКривий Ріг...,19318.4


Виглядає не дуже. Виправимо.

In [13]:
data[['year','people']].groupby('year').sum()

,people
year,
2001,21602.5
2014,20977.9
2020,19318.4


In [14]:
data.groupby('year').agg({'people':'sum', 'city':'max'})

,people,city
year,,
2001,21602.5,Ялта
2014,20977.9,Ялта
2020,19318.4,Ялта


In [15]:
part = data.iloc[:15, :]
part

,city,year,people
0,Київ,2001,2611.3
1,Київ,2014,2868.7
2,Київ,2020,2967.4
3,Харків,2001,1470.9
4,Харків,2014,1451.1
6,Одеса,2001,1029.0
7,Одеса,2014,1017.0
8,Одеса,2020,1017.7
9,Дніпро,2001,1080.8
10,Дніпро,2014,993.1


In [16]:
gr = part.groupby('year')
gr

In [17]:
gr.groups

{2001: [0, 3, 6, 9, 12, 15], 2014: [1, 4, 7, 10, 13], 2020: [2, 8, 11, 14]}

In [18]:
gr.groups[2001]

Index([0, 3, 6, 9, 12, 15], dtype='int64')

In [19]:
gr.indices

{np.int64(2001): array([ 0,  3,  5,  8, 11, 14]),
 np.int64(2014): array([ 1,  4,  6,  9, 12]),
 np.int64(2020): array([ 2,  7, 10, 13])}

In [20]:
gr.count()

,city,people
year,,
2001,6,6
2014,5,5
2020,4,4


In [21]:
gr.get_group(2020)

,city,year,people
2,Київ,2020,2967.4
8,Одеса,2020,1017.7
11,Дніпро,2020,990.7
14,Донецьк,2020,908.5


<b>Додаткові приклади групування та виконання групових (агрегатних) операцій над групами</b>.

In [22]:
data.groupby('city').count()

,year,people
city,,
Євпаторія,3,3
Єнакієве,3,3
Івано-Франківськ,3,3
Ізмаїл,3,3
Ірпінь,3,3
...,...,...
Чернігів,3,3
Чистякове,3,3
Чорноморськ,3,3


In [23]:
data.groupby('city').agg(['mean', 'count'])

year            people      
                         mean count        mean count
city                                                 
Євпаторія         2011.666667     3  107.033333     3
Єнакієве          2011.666667     3        87.5     3
Івано-Франківськ  2011.666667     3       227.7     3
Ізмаїл            2011.666667     3        76.2     3
Ірпінь            2011.666667     3   48.233333     3
...                       ...   ...         ...   ...
Чернігів          2011.666667     3  295.866667     3
Чистякове         2011.666667     3   61.066667     3
Чорноморськ       2011.666667     3   57.633333     3
Шостка            2011.666667     3        79.9     3
Ялта              2011.666667     3   79.666667     3

[87 rows x 4 columns]

In [24]:
data.groupby('city').agg({'year':'count', 'people':'mean'})

,year,people
city,,
Євпаторія,3,107.033333
Єнакієве,3,87.5
Івано-Франківськ,3,227.7
Ізмаїл,3,76.2
Ірпінь,3,48.233333
...,...,...
Чернігів,3,295.866667
Чистякове,3,61.066667
Чорноморськ,3,57.633333


In [25]:
data.groupby('city').agg({'year':'count', 'people':['mean', 'max']})

year      people       
                 count        mean    max
city                                     
Євпаторія            3  107.033333  108.2
Єнакієве             3        87.5  104.0
Івано-Франківськ     3       227.7  237.7
Ізмаїл               3        76.2   84.8
Ірпінь               3   48.233333   60.1
...                ...         ...    ...
Чернігів             3  295.866667  305.0
Чистякове            3   61.066667   72.3
Чорноморськ          3   57.633333   59.8
Шостка               3        79.9   87.1
Ялта                 3   79.666667   81.7

[87 rows x 3 columns]

In [26]:
data.pivot(index='city', columns='year', values='people')

year,2001,2014,2020
city,,,
Євпаторія,105.9,107.0,108.2
Єнакієве,104.0,81.1,77.4
Івано-Франківськ,218.4,227.0,237.7
Ізмаїл,84.8,72.5,71.3
Ірпінь,40.6,44.0,60.1
...,...,...,...
Чернігів,305.0,295.7,286.9
Чистякове,72.3,57.0,53.9
Чорноморськ,54.2,59.8,58.9
